# TCG-pokebot — resumable, persistent training on Kaggle

Kaggle is Linux x86-64, so the `cabt` engine loads directly. This notebook is built for the **12-hour kernel limit**:
it **checkpoints every iteration**, **stops cleanly before the kill**, and **resumes across sessions**, and it
**persists trained weights to a Kaggle Dataset** so they survive outside the notebook.

## Before you run — add these in the right-hand panel
1. **Add Input → Competitions**: the *Pokémon TCG AI Battle* competition (provides the engine `cg/`, `EN_Card_Data.csv`, sample `deck.csv`). You must have **joined** it.
2. **Add Input → Datasets** (for resume): your weights dataset `<your-username>/tcg-pokebot-weights` if it exists from a previous run. First run: skip this — the notebook bootstraps from scratch.
3. **Get the code** — pick ONE:
   - the repo `Hakase-0/TCG-pokebot` is **public** → nothing to do (cell 2 clones it; turn **Internet ON**), or
   - it's **private** → add a Kaggle **Secret** named `GITHUB_TOKEN` (a GitHub PAT with repo read) and Internet ON, or
   - **offline / no token** → upload the repo as a dataset and set `CODE_DATASET` in cell 1.
4. **Persistence** — add Kaggle **Secrets** `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com/settings/api) so cell 9 can push weights to a dataset. Without them it still saves to `/kaggle/working` (commit the notebook to keep them).

Secrets are set via **Add-ons → Secrets**. The notebook reads them through `UserSecretsClient`; your credentials are never written into the notebook.


## 1. Configuration — edit these

In [ ]:
USERNAME        = 'your-kaggle-username'   # <-- your Kaggle username (for the weights dataset slug)
REPO            = 'Hakase-0/TCG-pokebot'   # GitHub repo (public, or private w/ GITHUB_TOKEN secret)
CODE_DATASET    = ''                       # optional fallback: '<user>/tcg-pokebot-code' if offline
WEIGHTS_DATASET = f'{USERNAME}/tcg-pokebot-weights'  # where checkpoints are stored / resumed from

# --- training knobs ---
PHASE           = 'rollout'   # 'rollout' = strong engine-oracle teacher (~70s/game, accurate, slow)
                              # 'value'   = fast net value leaf (many more games/hr, weaker until distilled)
ITERS           = 6           # iterations to ATTEMPT this session (time budget may stop earlier)
GAMES           = 120         # self-play games per iteration
ISMCTS_WORLDS   = 2
ISMCTS_SIMS     = 24
GATE_GAMES      = 80          # promotion-gate games
FIELD_GAMES     = 12          # games/deck for the field check (raise for tighter CIs)
TIME_BUDGET_MIN = 660         # stop before starting an iter past this (Kaggle kills at 720)
OUR_DECK_SLUG   = 'dragapult-ex'   # which decks/<slug>.csv is OUR deck

import os, sys, glob, shutil, subprocess, time
WORKDIR = '/kaggle/working/TCG-pokebot'
print('config loaded; phase =', PHASE)


## 2. Get the code into a writable dir
Tries: private clone (GITHUB_TOKEN secret) → public clone → uploaded code dataset. Anchored on `ismcts.py` so a stale partial copy is re-fetched.

In [ ]:
def have_code(p):
    return os.path.exists(os.path.join(p, 'ismcts.py')) and os.path.exists(os.path.join(p, 'selfplay_rl.py'))

def _token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return None

if not have_code(WORKDIR):
    done = False
    tok = _token()
    url_pub = f'https://github.com/{REPO}'
    url_tok = f'https://{tok}@github.com/{REPO}' if tok else None
    for url in [u for u in (url_tok, url_pub) if u]:
        try:
            subprocess.run(['git','clone','--depth','1', url, WORKDIR], check=True)
            done = have_code(WORKDIR)
            if done:
                print('cloned from GitHub'); break
        except Exception as e:
            print('clone attempt failed:', str(e)[:120])
    if not done and CODE_DATASET:
        # fall back to an uploaded code dataset; copy (never rmtree the cwd)
        src = f'/kaggle/input/{CODE_DATASET.split("/")[-1]}'
        cand = next((d for d,_,f in os.walk(src) if 'ismcts.py' in f), None)
        if cand:
            os.makedirs(WORKDIR, exist_ok=True)
            for f in glob.glob(os.path.join(cand,'*')):
                (shutil.copytree if os.path.isdir(f) else shutil.copy)(f, os.path.join(WORKDIR, os.path.basename(f)))
            done = have_code(WORKDIR); print('copied from code dataset:', done)
    assert done, 'could not obtain code — see the setup notes in cell 0'
else:
    print('code already present')

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('cwd =', os.getcwd(), '| files:', len(glob.glob('*.py')))


## 3. Engine — copy `cg/` from the competition input and confirm it loads

In [ ]:
if not os.path.exists(os.path.join(WORKDIR,'cg','libcg.so')):
    so = next((os.path.join(d,'libcg.so') for d,_,fs in os.walk('/kaggle/input') if 'libcg.so' in fs), None)
    assert so, 'libcg.so not found under /kaggle/input — add the competition as input'
    shutil.copytree(os.path.dirname(so), os.path.join(WORKDIR,'cg'), dirs_exist_ok=True)
    print('copied engine from', os.path.dirname(so))
# card CSV + sample deck from the competition input (if not already local)
for fn in ('EN_Card_Data.csv',):
    if not os.path.exists(fn):
        src = next((os.path.join(d,fn) for d,_,fs in os.walk('/kaggle/input') if fn in fs), None)
        if src: shutil.copy(src, fn); print('copied', fn)
os.environ['CG_LIB_PATH'] = os.path.join(WORKDIR,'cg')
from cg import api
print('engine OK — cards:', len(api.all_card_data()))


## 4. Card tables (one-time per session) — emits evolution/role fields for deck inference

In [ ]:
subprocess.run([sys.executable,'inspect_cards.py'], check=True,
               env={**os.environ,'CG_LIB_PATH':os.path.join(WORKDIR,'cg')})
print('tables:', os.path.exists('capability_table.json'), os.path.exists('attack_table.json'))


## 5. Opponent deck pool — **offline** txt→id-CSV (no internet needed)
The shipped `decks/*.txt` are converted to id-CSVs via `EN_Card_Data.csv`. If Internet is ON you can instead refresh from Limitless with `build_deck_pool.py`.

In [ ]:
from import_deck import import_decklist, write_deck_csv
made = 0
for txt in sorted(glob.glob('decks/*.txt')) + sorted(glob.glob('decks/adversary/*.txt')):
    csv = txt[:-4] + '.csv'
    if os.path.exists(csv):
        made += 1; continue
    try:
        ids, _ = import_decklist(open(txt).read(), 'EN_Card_Data.csv')
        if len(ids) == 60:
            write_deck_csv(ids, csv); made += 1
    except Exception as e:
        print('skip', os.path.basename(txt), str(e)[:80])
meta = len(glob.glob('decks/*.csv')); adv = len(glob.glob('decks/adversary/*.csv'))
print(f'deck pool ready: {meta} meta + {adv} adversary id-CSVs')
# our deck.csv
our_src = f'decks/{OUR_DECK_SLUG}.csv'
if os.path.exists(our_src):
    shutil.copy(our_src, 'deck.csv'); print('our deck:', OUR_DECK_SLUG)
elif not os.path.exists('deck.csv'):
    raise SystemExit('no deck.csv and OUR_DECK_SLUG not found')


## 6. Resume — find the newest checkpoint to warm-start from
Looks in the attached weights dataset (and `/kaggle/working`). Prefers the continuously-trained `*.latest.pt`, then the gated best, then the BC `model.pt`.

In [ ]:
def find_ckpt():
    roots = [f'/kaggle/input/{WEIGHTS_DATASET.split("/")[-1]}', '/kaggle/working', WORKDIR]
    for pat in ('*.latest.pt','rl_model.pt','model.pt'):
        for r in roots:
            hits = sorted(glob.glob(os.path.join(r,'**',pat), recursive=True),
                          key=os.path.getmtime, reverse=True)
            if hits:
                return hits[0]
    return None

WARM = find_ckpt()
NEED_BC = WARM is None
print('resume from:', WARM if WARM else 'NONE — will bootstrap with BC (cell 7)')


## 7. (first run only) Behavioral-cloning bootstrap
Runs **only if no checkpoint was found**. Distills the engine-oracle combat agent into the net — the RL warm start. ~20–30 min.

In [ ]:
if NEED_BC:
    subprocess.run([sys.executable,'gen_selfplay_data.py','--games','300','--policy','combat',
                    '--out','data/bc.pkl','--log','logs/selfplay.jsonl'], check=True)
    subprocess.run([sys.executable,'train_bc.py','--data','data/bc.pkl','--epochs','20',
                    '--out','model.pt','--log','logs/train.jsonl'], check=True)
    WARM = 'model.pt'
    print('BC bootstrap complete -> model.pt')
else:
    print('skipping BC — resuming from', WARM)


## 8. Train — time-budgeted, checkpointed RL
Self-play uses ISMCTS with the chosen leaf. `--time-budget-min` stops cleanly before the 12h kill; the net is saved to `*.latest.pt` **every iteration** (crash-safe) and the gated best to `rl_model.pt`. Re-run the notebook in a new session to continue from where this left off.

In [ ]:
OUT = 'rl_model.pt'
LATEST = 'rl_model.latest.pt'
cmd = [sys.executable,'-u','selfplay_rl.py','--search','ismcts','--leaf',PHASE,
       '--ismcts-worlds',str(ISMCTS_WORLDS),'--ismcts-sims',str(ISMCTS_SIMS),
       '--iters',str(ITERS),'--games',str(GAMES),'--gate-games',str(GATE_GAMES),
       '--field-games',str(FIELD_GAMES),'--time-budget-min',str(TIME_BUDGET_MIN),
       '--warm',WARM,'--our-deck','deck.csv','--opp-decks','decks/',
       '--out',OUT,'--latest-out',LATEST,'--log','logs/rl.jsonl']
print(' '.join(cmd))
subprocess.run(cmd, check=True, env={**os.environ,'CG_LIB_PATH':os.path.join(WORKDIR,'cg')})
print('training segment done. checkpoints:', [f for f in (OUT,LATEST) if os.path.exists(f)])


## 8b. Sharpen the value head (minutes) + gate the fast leaf
The rollout run dumped value data as a byproduct. `fit_value.py` sharpens **only the value head** (frozen body -> policy unchanged) on the search-backed targets, in minutes. Then we A/B the **fast** value leaf vs the slow rollout leaf. **If value matches rollout within the CI, set `PHASE='value'`** — no engine rollouts in the search, so the 12h limit largely stops mattering.

In [ ]:
import glob
if glob.glob('valuedata/*.pkl') and os.path.exists('rl_model.pt'):
    subprocess.run([sys.executable,'fit_value.py','--data','valuedata','--warm','rl_model.pt',
                    '--out','rl_model.value.pt','--epochs','8','--target-blend','0.5'], check=True)
    print('\n--- A/B: does the FAST value leaf now match the SLOW rollout leaf? ---')
    for leaf in ('value','rollout'):
        print(f'\n[{leaf} leaf]')
        subprocess.run([sys.executable,'arena.py','--candidate','rl_model.value.pt','--anchor','rl_model.pt',
                        '--ismcts','--leaf',leaf,'--ismcts-worlds','2','--ismcts-sims','24',
                        '--games','100','--log','logs/arena.jsonl'],
                       env={**os.environ,'CG_LIB_PATH':os.path.join(WORKDIR,'cg')})
    print('\nIf value-leaf score >= rollout-leaf within CI: set PHASE=value (fast) and re-run.')
else:
    print('no value data / no rl_model.pt yet — run the rollout phase first')


## 9. Persist weights to a Kaggle Dataset (survives the notebook)
Pushes the checkpoints + logs as a **new version** of `WEIGHTS_DATASET` (creates it the first time) using your `KAGGLE_*` secrets. Always also copies to `/kaggle/working`, which is saved when you commit the notebook.

In [ ]:
import json
STAGE = '/kaggle/working/weights'
os.makedirs(STAGE, exist_ok=True)
for f in glob.glob('rl_model*.pt') + ['model.pt','model_meta.json'] + glob.glob('logs/*.jsonl'):
    if os.path.exists(f):
        shutil.copy(f, os.path.join(STAGE, os.path.basename(f)))
print('staged for output:', os.listdir(STAGE))

def _kaggle_creds():
    try:
        from kaggle_secrets import UserSecretsClient
        u = UserSecretsClient()
        os.environ['KAGGLE_USERNAME'] = u.get_secret('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = u.get_secret('KAGGLE_KEY')
        return True
    except Exception as e:
        print('no KAGGLE_* secrets -> dataset push skipped (weights still in /kaggle/working):', str(e)[:80])
        return False

if _kaggle_creds():
    slug = WEIGHTS_DATASET
    meta = {'title':'tcg-pokebot-weights','id':slug,
            'licenses':[{'name':'CC0-1.0'}]}
    json.dump(meta, open(os.path.join(STAGE,'dataset-metadata.json'),'w'))
    # try a new version; if the dataset doesn't exist yet, create it
    msg = f'weights {time.strftime("%Y-%m-%d %H:%M")} phase={PHASE}'
    r = subprocess.run(['kaggle','datasets','version','-p',STAGE,'-m',msg,'--dir-mode','zip'],
                       capture_output=True, text=True)
    if r.returncode != 0 and 'not found' in (r.stdout+r.stderr).lower():
        r = subprocess.run(['kaggle','datasets','create','-p',STAGE,'--dir-mode','zip'],
                           capture_output=True, text=True)
    print(r.stdout[-400:] or r.stderr[-400:])
    print('-> add', slug, 'as an input next session to resume')


## 10. (optional) Measured baseline — is the RL net beating BC?
Runs only after a full session. Wide CIs at low game counts — read the aggregate, not per-deck cells.

In [ ]:
if os.path.exists('rl_model.pt'):
    subprocess.run([sys.executable,'arena.py','--candidate','rl_model.pt','--anchor','model.pt',
                    '--games','200','--field','--adversary','--log','logs/arena.jsonl'],
                   env={**os.environ,'CG_LIB_PATH':os.path.join(WORKDIR,'cg')})
else:
    print('no rl_model.pt yet')
